In [1]:
# Google Colab cell for cloud API

# Install dependencies
!pip install fastapi uvicorn unsloth sentence-transformers joblib torch spacy transformers huggingface-hub pydantic numpy pyngrok
! pip install huggingface_hub[hf_xet]

# Install spaCy model
!python -m spacy download en_core_web_sm

# Install ngrok
# !pip install pyngrok

# ! curl -X POST  https://7ad2-35-197-106-150.ngrok-free.app/generate_sentences \
#      -H "Content-Type: application/json" \
#      -d '{"level": "A1", "num_sentences": 2, "user_id": 1}'

# curl -X POST https://b84d-34-124-230-90.ngrok-free.app/generate_sentences \
#      -H "Content-Type: application/json" \
#      -d '{"level": "A1", "num_sentences": 2, "user_id": 1, "topic": "food"}'


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
/home/citnlp/yechiel/bin/python: No module named spacy


In [ ]:
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel
import uvicorn
from unsloth import FastLanguageModel
from peft import PeftModel
from sentence_transformers import SentenceTransformer
import joblib
import numpy as np
import re
import random
from huggingface_hub import snapshot_download
import torch
import spacy
from pyngrok import ngrok
import nest_asyncio
import threading
import asyncio
import time
from typing import List, Dict, Optional
from transformers import StoppingCriteria
import datetime
import os

# Allow nested event loops for Colab
nest_asyncio.apply()

# Define FastAPI app
app = FastAPI()

# Global variables for loaded models and constants
GENERATOR_MODEL = None
GENERATOR_TOKENIZER = None
DISTRACTOR_MODEL = None
DISTRACTOR_TOKENIZER = None
EVALUATOR_MODEL = None
EMBEDDER_MODEL = None
SPACY_MODEL = None
MAX_SEQ_LENGTH = 2048

# Define request models
class SentenceRequest(BaseModel):
    level: str
    num_sentences: int = 1
    user_id: int
    topic: Optional[str] = None
    question_types: Optional[List[str]] = ["selection", "labeling", "fill_in_blank", "multiple_choice", "arrange"]

class SelectionRequest(BaseModel):
    selected_sentence: str
    sentences: str  # Changed to str to match the | delimiter
    correct_sentence: str
    target_level: str
    user_id: int

class LabelingRequest(BaseModel):
    sentence: str
    user_labels: Dict[str, str]
    target_level: str
    user_id: int

class FillInBlankRequest(BaseModel):
    sentence: str
    user_answer: str
    correct_answer: str
    target_level: str
    user_id: int

class MultipleChoiceRequest(BaseModel):
    selected_option: str
    options: List[str]
    correct_answer: str
    target_level: str
    user_id: int

class ArrangeRequest(BaseModel):
    sentence: str
    user_arrangement: List[str]
    correct_sentence: str
    target_level: str
    user_id: int

# Define response models
class SentenceResponse(BaseModel):
    success: bool
    sentences: List[Dict]
    message: str
    processing_time: float
    timestamp: str

class SelectionResponse(BaseModel):
    success: bool
    is_correct: bool
    score: float
    message: str
    processing_time: float
    timestamp: str

class LabelingResponse(BaseModel):
    success: bool
    is_correct: bool
    score: float
    message: str
    processing_time: float
    timestamp: str

class FillInBlankResponse(BaseModel):
    success: bool
    is_correct: bool
    score: float
    message: str
    processing_time: float
    timestamp: str

class MultipleChoiceResponse(BaseModel):
    success: bool
    is_correct: bool
    score: float
    message: str
    processing_time: float
    timestamp: str

class ArrangeResponse(BaseModel):
    success: bool
    is_correct: bool
    score: float
    message: str
    processing_time: float
    timestamp: str

class GrammarCorrectionRequest(BaseModel):
    user_id: int
    sentence: str

# Define response model
class GrammarCorrectionResponse(BaseModel):
    success: bool
    user_id: int
    original: str
    corrected: str
    corrections: List[str]

class StopOnTokens(StoppingCriteria):
    def __init__(self, tokenizer, stop_strings):
        self.tokenizer = tokenizer
        self.stop_strings = stop_strings
        self.stop_ids = [tokenizer.encode(s, add_special_tokens=False) for s in stop_strings]

    def __call__(self, input_ids, scores, **kwargs):
        for stop_id in self.stop_ids:
            if len(input_ids[0]) >= len(stop_id):
                if all(input_ids[0][-len(stop_id):] == stop_id):
                    return True
        return False

# Model loading function
def load_generator_models():
    global GENERATOR_MODEL, GENERATOR_TOKENIZER, DISTRACTOR_MODEL, DISTRACTOR_TOKENIZER, EVALUATOR_MODEL, EMBEDDER_MODEL, SPACY_MODEL
    print("Loading all models...")
    dtype = None
    load_in_4bit = True
    # generator_model_name = "Mr-FineTuner/Test_02_llama_1epoch_trainPercen_myValidator_fix"
    generator_model_name = "Mr-FineTuner/With_synthetic_Dataset_llama-1epoch"
    try:
        GENERATOR_MODEL, GENERATOR_TOKENIZER = FastLanguageModel.from_pretrained(
            model_name=generator_model_name,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=dtype,
            load_in_4bit=load_in_4bit,
        )
        if GENERATOR_TOKENIZER.pad_token is None:
            GENERATOR_TOKENIZER.pad_token = GENERATOR_TOKENIZER.eos_token
        GENERATOR_MODEL = FastLanguageModel.for_inference(GENERATOR_MODEL)
        print(f"Finetuned generator model '{generator_model_name}' loaded successfully!")
    except Exception as e:
        print(f"Error loading finetuned generator model: {str(e)}")
        raise
    print("Loading plain distractor model...")
    distractor_model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
    try:
        DISTRACTOR_MODEL, DISTRACTOR_TOKENIZER = FastLanguageModel.from_pretrained(
            model_name=distractor_model_name,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=dtype,
            load_in_4bit=load_in_4bit,
        )
        if DISTRACTOR_TOKENIZER.pad_token is None:
            DISTRACTOR_TOKENIZER.pad_token = DISTRACTOR_TOKENIZER.eos_token
        DISTRACTOR_MODEL = FastLanguageModel.for_inference(DISTRACTOR_MODEL)
        print(f"Plain distractor model '{distractor_model_name}' loaded successfully!")
    except Exception as e:
        print(f"Error loading plain distractor model: {str(e)}")
        raise
    print("Loading validator model...")
    try:
        eval_model_path = snapshot_download(
            repo_id="Mr-FineTuner/Skripsi_validator_best_model",
            allow_patterns=["model_mlp.joblib"],
            local_dir="validator_model"
        )
        classifier_path = os.path.join(eval_model_path, "model_mlp.joblib")
        if not os.path.exists(classifier_path):
            raise FileNotFoundError(f"model_mlp.joblib not found in {eval_model_path}")
        EVALUATOR_MODEL = joblib.load(classifier_path)
        EMBEDDER_MODEL = SentenceTransformer('BAAI/bge-base-en-v1.5')
        print("Validator models loaded successfully!")
    except Exception as e:
        print(f"Error loading validator models: {str(e)}")
        raise
    print("Loading spaCy model...")
    try:
        SPACY_MODEL = spacy.load("en_core_web_sm")
        print("spaCy model loaded successfully!")
    except Exception as e:
        print(f"Error loading spaCy model: {str(e)}")
        raise

# Sentence generation logic
def generate_sentences_internal(level: str, num_sentences: int, user_id: int, topic: Optional[str] = None, question_types: List[str] = ["selection", "labeling", "fill_in_blank", "multiple_choice", "arrange"]) -> List[Dict]:
    global GENERATOR_MODEL, GENERATOR_TOKENIZER, DISTRACTOR_MODEL, DISTRACTOR_TOKENIZER, EVALUATOR_MODEL, EMBEDDER_MODEL, SPACY_MODEL
    if None in [GENERATOR_MODEL, GENERATOR_TOKENIZER, DISTRACTOR_MODEL, DISTRACTOR_TOKENIZER, EVALUATOR_MODEL, EMBEDDER_MODEL, SPACY_MODEL]:
        raise ValueError("One or more models failed to load")

    # Contraction mapping for expansion
    CONTRACTION_MAP = {
        "'ve": "have",
        "'s": "is",
        "'re": "are",
        "'m": "am",
        "'ll": "will",
        "'d": "had"
    }

    error_counts = {
        "duplicate_sentence": 0,
        "cefr_mismatch": 0,
        "no_valid_verb": 0,
        "no_valid_constituents": 0,
        "insufficient_constituents": 0,
        "multiple_clauses": 0,
        "passive_voice": 0,
        "output_pattern_removed": 0,
        "duplicate_selection": 0,
        "invalid_incorrect_sentence": 0,
        "invalid_fill_in_blank": 0,
        "invalid_multiple_choice": 0,
        "invalid_arrange": 0,
        "no_full_stop": 0
    }

    def predict_sentiment(text: str) -> int:
        embedding = EMBEDDER_MODEL.encode([text])
        probs = EVALUATOR_MODEL.predict_proba(embedding)[0]
        prediction = np.argmax(probs) + 1
        return max(1, min(6, int(prediction)))

    cefr_to_score = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
    score_to_cefr = {1: 'A1', 2: 'A2', 3: 'B1', 4: 'B2', 5: 'C1', 6: 'C2'}

    def clean_sentence(text: str) -> Optional[str]:
        cleaned = re.sub(r'\n+#+\s*Output\s*:', '', text, flags=re.IGNORECASE)
        cleaned = re.sub(r'\n+\s*###\s*Output\s*:', '', cleaned, flags=re.IGNORECASE)
        cleaned = cleaned.strip()
        match = re.match(r'^(.*?)\.', cleaned)
        if match:
            cleaned = match.group(1) + '.'
        else:
            print("Debug: No full-stop found in sentence, rejecting.")
            return None
        cleaned = cleaned.strip()
        if not cleaned:
            print("Debug: Cleaned sentence is empty, rejecting.")
            return None
        return cleaned

    def is_article_correct(sent: str, orig_doc) -> bool:
        doc = SPACY_MODEL(sent)
        for i, token in enumerate(doc):
            if token.dep_ == "nsubj" and i > 0:
                prev = doc[i-1].text.lower()
                if prev in ["a", "an"]:
                    first_letter = orig_doc[i].text[0].lower()
                    is_vowel = first_letter in 'aeiou'
                    if (prev == "a" and is_vowel) or (prev == "an" and not is_vowel):
                        return False
                elif prev == "the":
                    return False
            elif token.dep_ == "nsubj" and i == 0:
                if token.tag_ in ["NN", "NNP"]:
                    return False
        return True

    def is_subject_verb_agreement_correct(sent: str) -> bool:
        doc = SPACY_MODEL(sent)
        subject = None
        verb = None
        for token in doc:
            if token.dep_ == "nsubj":
                subject = token
            if token.dep_ == "ROOT" and token.pos_ in ["VERB", "AUX"]:
                verb = token
        if subject and verb:
            is_subject_plural = subject.tag_ in ["NNS", "NNPS"]
            is_verb_singular = verb.text.lower() in ["is", "was", "has"]
            return not (is_subject_plural != is_verb_singular)
        return True

    def create_selection_question(sentence: str, level: str) -> Dict:
        error_types = {
            'A1': ["spelling", "article", "number_agreement"],
            'A2': ["spelling", "article", "tense"],
            'B1': ["tense", "preposition", "word_choice"],
            'B2': ["tense", "preposition", "agreement"],
            'C1': ["complex_tense", "idiom", "word_form"],
            'C2': ["complex_tense", "idiom", "subjunctive"]
        }
        available_errors = error_types.get(level, error_types['B1'])
        if len(available_errors) < 2:
            available_errors = available_errors + ["spelling"]
        errors = random.sample(available_errors, 2)

        doc = SPACY_MODEL(sentence)
        words = [token.text for token in doc]

        def introduce_error(sent: str, error_type: str) -> str:
            doc = SPACY_MODEL(sent)
            words = [token.text for token in doc]
            modified = sent
            if error_type == "spelling":
                for i, word in enumerate(words):
                    if len(word) > 3 and word.isalpha():
                        if len(word) > 4:
                            words[i] = word[:-2] + random.choice('ae') + random.choice('mn')
                        else:
                            words[i] = word[:-1] + random.choice('ae')
                        modified = " ".join(words).strip()
                        if SPACY_MODEL(words[i])[0].is_oov:
                            break
                else:
                    modified = sent
            elif error_type == "article":
                for i, token in enumerate(doc):
                    if token.dep_ == "nsubj" and i > 0 and words[i-1].lower() in ["a", "an"]:
                        words[i-1] = ""
                        modified = " ".join(words).strip()
                        break
                    elif token.dep_ == "nsubj":
                        first_letter = token.text[0].lower()
                        is_vowel = first_letter in 'aeiou'
                        words[i] = f"{'a' if is_vowel else 'an'} {words[i]}"
                        modified = " ".join(words).strip()
                        break
                else:
                    modified = sent
            elif error_type == "number_agreement":
                for i, token in enumerate(doc):
                    if token.dep_ == "nsubj":
                        is_plural = token.tag_ in ["NNS", "NNPS"]
                        if is_plural:
                            words[i] = SPACY_MODEL(token.lemma_)[0].text
                        else:
                            words[i] = token.text + "s"
                        for j, t in enumerate(doc):
                            if t.dep_ == "ROOT" and t.text.lower() in ["is", "are"]:
                                words[j] = "are" if t.text.lower() == "is" else "is"
                                break
                        modified = " ".join(words).strip()
                        break
                else:
                    modified = sent
            elif error_type == "tense":
                for i, token in enumerate(doc):
                    if token.pos_ == "VERB" and token.text.lower() not in ["be", "can"]:
                        words[i] = SPACY_MODEL(token.lemma_)[0].text + "ed"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "preposition":
                for i, token in enumerate(doc):
                    if token.dep_ == "prep":
                        words[i] = "to" if token.text != "to" else "in"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "word_choice":
                for i, token in enumerate(doc):
                    if token.dep_ == "dobj":
                        words[i] = "banana" if token.text.lower() != "banana" else "orange"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "agreement":
                for i, token in enumerate(doc):
                    if token.dep_ == "nsubj" and token.text.lower() == "i":
                        words[i] = "We"
                        for v in doc:
                            if v.pos_ == "VERB" and v.text.lower() in ["like", "eat"]:
                                words[v.i] = v.text + "s"
                                break
                        modified = " ".join(words).strip()
                        break
            elif error_type == "complex_tense":
                for i, token in enumerate(doc):
                    if token.pos_ == "VERB":
                        words[i] = f"had {token.lemma_}ed"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "idiom":
                for i, token in enumerate(doc):
                    if token.dep_ == "dobj":
                        words[i] = "moon"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "word_form":
                for i, token in enumerate(doc):
                    if token.pos_ == "VERB":
                        words[i] = token.lemma_ + "ing"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "subjunctive":
                for i, token in enumerate(doc):
                    if token.pos_ == "VERB":
                        words[i] = token.lemma_
                        modified = " ".join(words).strip()
                        break
            modified = clean_sentence(modified)
            if modified is None or modified.strip() == sent.strip():
                for i, word in enumerate(words):
                    if len(word) > 3 and word.isalpha():
                        words[i] = word[:-1] + random.choice('ae')
                        modified = clean_sentence(" ".join(words).strip())
                        break
            return modified if modified else sent

        incorrect1 = introduce_error(sentence, errors[0])
        incorrect2 = introduce_error(sentence, errors[1])
        max_error_retries = 3
        error_retry_count = 0
        while (incorrect2 == incorrect1 or incorrect2 == sentence) and error_retry_count < max_error_retries:
            new_error = random.choice([e for e in available_errors if e != errors[0]])
            incorrect2 = introduce_error(sentence, new_error)
            errors[1] = new_error
            error_retry_count += 1
        if incorrect2 == incorrect1 or incorrect2 == sentence:
            for i, word in enumerate(words):
                if len(word) > 3 and word.isalpha():
                    words[i] = word[:-1] + 'u'
                    incorrect2 = clean_sentence(" ".join(words).strip())
                    errors[1] = "spelling"
                    break
        sentences = [sentence, incorrect1, incorrect2]
        for i, s in enumerate(sentences):
            if s != sentence:
                is_valid = True
                try:
                    if errors[i-1] == "article" and is_article_correct(s, doc):
                        is_valid = False
                    elif errors[i-1] == "number_agreement" and is_subject_verb_agreement_correct(s):
                        is_valid = False
                    elif errors[i-1] == "spelling":
                        s_doc = SPACY_MODEL(s)
                        for token, orig_token in zip(s_doc, doc):
                            if token.text != orig_token.text and not token.is_oov:
                                is_valid = False
                                break
                except Exception as e:
                    print(f"Debug: Validation error for '{s}': {str(e)}")
                    is_valid = False
                if not is_valid:
                    print(f"Debug: Invalid incorrect sentence '{s}' for error '{errors[i-1]}'")
                    error_counts["invalid_incorrect_sentence"] += 1
                    new_error = random.choice([e for e in available_errors if e != errors[i-1]])
                    sentences[i] = introduce_error(sentence, new_error)
                    errors[i-1] = new_error
        if len(set(sentences)) != 3:
            print(f"Debug: Duplicate sentences detected in selection question: {sentences}")
            error_counts["duplicate_selection"] += 1
            incorrect2 = introduce_error(sentence, random.choice(available_errors))
            sentences = [sentence, incorrect1, incorrect2]
            if len(set(sentences)) != 3:
                for i, word in enumerate(words):
                    if len(word) > 3 and word.isalpha():
                        words[i] = word[:-1] + 'u'
                        incorrect2 = clean_sentence(" ".join(words).strip())
                        sentences = [sentence, incorrect1, incorrect2]
                        break
        random.shuffle(sentences)
        return {
            "question_text": "Choose the correct sentence.",
            "sentences": "|".join(sentences),  # Use | as delimiter
            "correct_sentence": sentence
        }

    def create_fill_in_blank(sentence: str, level: str) -> Optional[Dict]:
        doc = SPACY_MODEL(sentence)
        verb = None
        verb_index = None
        verb_lemma = None
        for i, token in enumerate(doc):
            if token.pos_ in ["VERB", "AUX"] and token.dep_ in ["ROOT", "cop"]:
                if token.text.lower() not in ["to"]:
                    verb = token.text
                    verb_index = i
                    verb_lemma = token.lemma_.lower()
                    break
        if not verb or not verb_lemma:
            print("Debug: No valid verb for fill-in-the-blank, skipping.")
            error_counts["invalid_fill_in_blank"] += 1
            return None
        words = [token.text for token in doc]
        words[verb_index] = "__"
        blanked_sentence = " ".join(words).strip()
        question_text = f"{blanked_sentence[:-1]} ({verb_lemma})."
        return {
            "question_text": question_text,
            "correct_answer": verb
        }

    def generate_contextual_distractors(target_verb: str, sentence: str, num_distractors: int = 3) -> List[str]:
        print(f"Debug: Starting distractor generation for verb '{target_verb}' in sentence '{sentence}'")
        target_doc = SPACY_MODEL(target_verb)
        target_tense = None
        for token in target_doc:
            if token.pos_ == "VERB":
                target_tense = token.tag_
                break
        print(f"Debug: Target verb tense tag: {target_tense if target_tense else 'None'}")
        tense_map = {
            'VB': 'base form',
            'VBD': 'past tense',
            'VBG': 'gerund/present participle',
            'VBN': 'past participle',
            'VBP': 'present tense (non-3rd person singular)',
            'VBZ': 'present tense (3rd person singular)'
        }
        tense_desc = tense_map.get(target_tense, 'base form') if target_tense else 'base form'
        print(f"Debug: Tense description for prompt: {tense_desc}")
        total_distractors_to_generate = 6
        prompt = (
            f"Generate exactly {total_distractors_to_generate} English verbs that are distinct from '{target_verb}', "
            f"semantically related, and appropriate for CEFR level {level}. "
            f"The verbs must fit the context of the sentence '{sentence}' "
            f"and be in the {tense_desc} (e.g., '{target_verb}'). "
            f"Return only the verbs as a comma-separated list, with no additional text or non-verbs."
        )
        print(f"Debug: LLM prompt:\n{prompt}")
        try:
            input_data = DISTRACTOR_TOKENIZER(
                prompt,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).to("cuda" if torch.cuda.is_available() else "cpu")
            print(f"Debug: Tokenized input data: {input_data}")
            outputs = DISTRACTOR_MODEL.generate(
                input_ids=input_data["input_ids"],
                attention_mask=input_data["attention_mask"],
                max_new_tokens=50,
                pad_token_id=DISTRACTOR_TOKENIZER.pad_token_id,
                do_sample=True,
                temperature=0.6,
                top_k=50,
                top_p=0.85,
            )
            print(f"Debug: Raw LLM output: {outputs}")
            generated_text = DISTRACTOR_TOKENIZER.decode(outputs[0], skip_special_tokens=True)
            print(f"Debug: Decoded LLM output: '{generated_text}'")
            response_start = generated_text.find("<|start_header_id|>assistant<|end_header_id>") + len("<|start_header_id|>assistant<|end_header_id>")
            response_text = generated_text[response_start:].strip()
            print(f"Debug: Extracted response text: '{response_text}'")
            distractors = [v.strip() for v in response_text.split(",") if v.strip()]
            print(f"Debug: Parsed distractors: {distractors}")
            primary_distractors = []
            backup_distractors = []
            for verb in distractors:
                if not (len(verb.split()) == 1 and re.match(r'^[a-zA-Z]+$', verb)):
                    print(f"Debug: Invalid distractor '{verb}' - not a single alphabetic word")
                    continue
                if verb.lower() == target_verb.lower():
                    print(f"Debug: Invalid distractor '{verb}' - matches target verb")
                    continue
                doc = SPACY_MODEL(verb)
                is_verb = any(token.pos_ == "VERB" for token in doc)
                print(f"Debug: Validating '{verb}' - is_verb: {is_verb}")
                if not is_verb:
                    print(f"Debug: Invalid distractor '{verb}' - not recognized as a verb")
                    continue
                if len(primary_distractors) < num_distractors:
                    primary_distractors.append(verb)
                elif len(backup_distractors) < num_distractors:
                    backup_distractors.append(verb)
                if len(primary_distractors) >= num_distractors and len(backup_distractors) >= num_distractors:
                    break
            print(f"Debug: Primary distractors: {primary_distractors}")
            print(f"Debug: Backup distractors: {backup_distractors}")
            selected_distractors = primary_distractors[:num_distractors]
            if len(selected_distractors) < num_distractors:
                print(f"Debug: Insufficient primary distractors ({len(selected_distractors)}/{num_distractors}). Using backups.")
                needed = num_distractors - len(selected_distractors)
                valid_backups = [v for v in backup_distractors if v not in selected_distractors]
                selected_distractors.extend(valid_backups[:needed])
                print(f"Debug: Added backups: {valid_backups[:needed]}")
            if len(selected_distractors) < num_distractors:
                print(f"Debug: Insufficient distractors after backups ({len(selected_distractors)}/{num_distractors}). Using fallbacks.")
                fallback_verbs = ["walk", "talk", "look", "move", "say"]
                fallback_verbs = [v for v in fallback_verbs if v.lower() != target_verb.lower() and v not in selected_distractors]
                selected_distractors.extend(fallback_verbs[:num_distractors - len(selected_distractors)])
                print(f"Debug: Added fallbacks: {selected_distractors[len(selected_distractors) - (num_distractors - len(selected_distractors)):]}")
            print(f"Debug: Final distractors: {selected_distractors[:num_distractors]}")
            return selected_distractors[:num_distractors]
        except Exception as e:
            print(f"Debug: Error generating distractors with LLM: {str(e)}")
            print(f"Debug: Using fallback distractors due to error")
            fallback_verbs = ["walk", "talk", "look"]
            return fallback_verbs[:num_distractors]

    def create_multiple_choice(sentence: str, level: str) -> Optional[Dict]:
        doc = SPACY_MODEL(sentence)
        verb = None
        verb_index = None
        for i, token in enumerate(doc):
            if token.pos_ in ["VERB", "AUX"] and token.dep_ in ["ROOT", "cop"]:
                if token.text.lower() not in ["to"]:
                    verb = token.text
                    verb_index = i
                    break
        if not verb:
            print("Debug: No valid verb for multiple choice, skipping.")
            error_counts["invalid_multiple_choice"] += 1
            return None
        words = [token.text for token in doc]
        words[verb_index] = "__"
        blanked_sentence = " ".join(words).strip()
        distractors = generate_contextual_distractors(verb, sentence)
        options = distractors + [verb]
        cleaned_options = []
        for opt in options:
            clean_opt = re.sub(r'[^a-zA-Z]', '', opt.lower()).strip()
            if clean_opt and len(clean_opt.split()) == 1:
                cleaned_options.append(clean_opt)
        if len(cleaned_options) < 4:
            print("Debug: Not enough valid options, returning None")
            error_counts["invalid_multiple_choice"] += 1
            return None
        random.shuffle(cleaned_options)
        return {
            "question_text": blanked_sentence,
            "options": cleaned_options,
            "correct_answer": verb.lower()
        }

    def create_arrange_question(sentence: str, level: str) -> Optional[Dict]:
        doc = SPACY_MODEL(sentence)
        words = [token.text for token in doc if token.text != '.']
        if len(words) < 3:
            print("Debug: Sentence too short for arrange question, skipping.")
            error_counts["invalid_arrange"] += 1
            return None
        random.shuffle(words)
        if ' '.join(words).strip().lower() == sentence.strip().lower():
            random.shuffle(words)
        return {
            "question_text": "Arrange the words to form the correct sentence.",
            "words": words,
            "correct_sentence": sentence
        }

    def create_labeling_question(sentence: str, level: str) -> Optional[Dict]:
        cleaned_sentence = clean_sentence(sentence)
        if cleaned_sentence is None:
            print("Debug: Invalid sentence for labeling question, skipping.")
            return None
        doc = SPACY_MODEL(cleaned_sentence)
        root_count = sum(1 for token in doc if token.dep_ == "ROOT")
        if root_count > 1:
            print("Debug: Multiple clauses detected, skipping labeling question.")
            error_counts["multiple_clauses"] += 1
            return None
        is_passive = any(token.dep_ == "auxpass" for token in doc)
        if is_passive:
            print("Debug: Passive voice detected, skipping labeling question.")
            error_counts["passive_voice"] += 1
            return None
        constituents = {
            "subject": None,
            "verb": None,
            "object": None,
            "adjective": None,
            "adverb": None,
            "prepositional phrase": None
        }
        prep_obj = None
        main_verb = None
        for token in doc:
            if token.dep_ == "nsubj" and not constituents["subject"]:
                constituents["subject"] = token.text
                for child in token.children:
                    if child.dep_ in ["det", "compound"]:
                        constituents["subject"] = f"{child.text} {constituents['subject']}"
            if token.dep_ == "ROOT" and token.pos_ in ["VERB", "AUX"]:
                if token.pos_ == "AUX" and not main_verb:
                    for child in token.children:
                        if child.pos_ == "VERB" and child.dep_ in ["xcomp", "ccomp", "advcl"]:
                            main_verb = child.text
                            break
                if not main_verb:
                    main_verb = token.text
            if token.dep_ == "dobj" and not constituents["object"]:
                obj = token.text
                for child in token.children:
                    if child.dep_ in ["det", "amod", "compound"]:
                        obj = f"{child.text} {obj}"
                constituents["object"] = obj
            if token.dep_ in ["amod", "acomp"] and token.pos_ == "ADJ" and not constituents["adjective"]:
                constituents["adjective"] = token.text
            if token.dep_ == "advmod" and token.pos_ == "ADV" and not constituents["prepositional phrase"]:
                for child in token.head.children:
                    if child.dep_ == "prep" and child.text in ["around", "about"]:
                        constituents["prepositional phrase"] = f"{child.text}"
                        for gc in child.children:
                            if gc.dep_ == "pobj":
                                constituents["prepositional phrase"] += f" {gc.text}"
                                break
                        print(f"Debug: Relabeled '{token.text}' as prepositional phrase")
                        break
                else:
                    constituents["adverb"] = token.text
            if token.dep_ == "prep" and not constituents["prepositional phrase"]:
                for child in token.children:
                    if child.dep_ == "pobj":
                        prep_obj = child.text
                        modifiers = []
                        for grandchild in child.children:
                            if grandchild.dep_ in ["det", "amod", "nummod", "compound"]:
                                modifiers.append(grandchild.text)
                        modifiers = sorted(modifiers, key=lambda x: doc.text.find(x))
                        prep_obj = " ".join(modifiers + [prep_obj]).strip()
                        constituents["prepositional phrase"] = f"{token.text} {prep_obj}"
                        break
        constituents["verb"] = main_verb
        if constituents["verb"]:
            for contraction, expansion in CONTRACTION_MAP.items():
                if contraction in constituents["verb"].lower():
                    constituents["verb"] = expansion
        valid_constituents = [k for k, v in constituents.items() if v is not None]
        if len(valid_constituents) < 3:
            print(f"Debug: Insufficient constituents ({len(valid_constituents)}/3): " +
                  ", ".join(f"{k} ({v if v else 'None'})" for k, v in constituents.items()) +
                  ", skipping labeling question.")
            error_counts["insufficient_constituents"] += 1
            return None
        valid_values = [v.lower() for v in constituents.values() if v is not None]
        if len(set(valid_values)) != len(valid_values):
            print("Debug: Constituents are not distinct, skipping labeling question.")
            error_counts["no_valid_constituents"] += 1
            return None
        correct_labels = {k: v for k, v in constituents.items() if v is not None}
        return {
            "question_text": cleaned_sentence,
            "instruction": f"Assign {', '.join(correct_labels.keys())} to the correct words or phrases.",
            "correct_labels": correct_labels
        }

    validated_sentences = []
    target_score = cefr_to_score.get(level, 1)
    attempt_count = 0
    max_attempts = 200

    while len(validated_sentences) < num_sentences and attempt_count < max_attempts:
        attempt_count += 1
        print(f"Debug: Generating sentence attempt {attempt_count}/{max_attempts} for level {level}")
        try:
            input_text = f"### Input:\nGenerate a sentence at CEFR level {level}"
            if topic:
                input_text += f" about {topic}"
            input_text += " Please don't use to be Verb and just output the sentence\n### Response:\n"
            input_data = GENERATOR_TOKENIZER(
                input_text,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).to("cuda" if torch.cuda.is_available() else "cpu")
            max_tokens_map = {
                'A1': 30,
                'A2': 30,
                'B1': 35,
                'B2': 35,
                'C1': 50,
                'C2': 60
            }
            max_new_tokens = max_tokens_map.get(level, 30)
            print(f"Debug: Using max_new_tokens={max_new_tokens} for level {level}")
            outputs = GENERATOR_MODEL.generate(
                input_ids=input_data["input_ids"],
                attention_mask=input_data["attention_mask"],
                max_new_tokens=max_new_tokens,
                pad_token_id=GENERATOR_TOKENIZER.pad_token_id,
                do_sample=True,
                temperature=0.8,
                top_k=60,
                top_p=0.9,
            )
            if not isinstance(outputs, torch.Tensor) or outputs.numel() == 0:
                print("Debug: Invalid model output, skipping.")
                continue
            generated_text = GENERATOR_TOKENIZER.decode(outputs[0], skip_special_tokens=True)
            if not isinstance(generated_text, str) or not generated_text.strip():
                print("Debug: Invalid decoded text, skipping.")
                continue
            response_start = generated_text.find("### Response:\n") + len("### Response:\n")
            response_text = generated_text[response_start:].strip()
            response_text = re.sub(r'^Example sentence at level [A-C][1-2]: ', '', response_text, flags=re.IGNORECASE)
            original_response = response_text
            print(f"Debug: Raw model output: '{original_response}'")
            response_text = clean_sentence(response_text)
            if response_text is None:
                error_counts["no_full_stop"] += 1
                print("Debug: Sentence rejected due to missing full-stop, retrying.")
                continue
            print(f"Debug: Original sentence: '{original_response}'")
            print(f"Debug: Cleaned sentence: '{response_text}'")
            if original_response != response_text:
                print(f"Debug: Processed sentence: '{original_response}' -> '{response_text}'")
                error_counts["output_pattern_removed"] += 1
            for contraction, expansion in CONTRACTION_MAP.items():
                response_text = re.sub(rf'\b(\w+){contraction}\b', r'\1 ' + expansion, response_text, flags=re.IGNORECASE)
            doc = SPACY_MODEL(response_text)
            sents = list(doc.sents)
            first_sent = sents[0].text.strip() if sents else response_text
            if len(first_sent.split()) < 3:
                print("Debug: Sentence too short, skipping.")
                continue
            doc = SPACY_MODEL(first_sent)
            verbs = []
            for token in doc:
                if token.pos_ in ["VERB", "AUX"] and token.dep_ in ["ROOT", "cop", "aux"]:
                    if token.text.lower() == "to" and token.dep_ == "aux":
                        continue
                    if token.pos_ == "AUX" and token.dep_ == "ROOT":
                        for child in token.children:
                            if child.pos_ == "VERB" and child.dep_ in ["xcomp", "ccomp", "advcl"]:
                                verbs.append(child.text)
                                print(f"Debug: Selected main verb '{child.text}' over modal '{token.text}'")
                                break
                        else:
                            verbs.append(token.text)
                    else:
                        verbs.append(token.text)
            if not verbs and "'" in first_sent:
                contractions = [token for token in doc if "'" in token.text and token.pos_ in ["VERB", "AUX"]]
                if contractions:
                    verbs.append(contractions[0].text)
            verb = verbs[0] if verbs else None
            if not verb:
                error_counts["no_valid_verb"] += 1
                print("Debug: No valid verb found, skipping.")
                continue
            predicted_score = predict_sentiment(first_sent)
            if abs(predicted_score - target_score) > 1:
                error_counts["cefr_mismatch"] += 1
                print("Debug: Sentence CEFR level mismatch, skipping.")
                continue
            if any(existing["sentence"] == first_sent for existing in validated_sentences):
                error_counts["duplicate_sentence"] += 1
                print("Debug: Duplicate sentence, skipping.")
                continue
            sentence_data = {
                "sentence": first_sent,
                "verb": verb,
                "level": level,
                "for_user": user_id
            }
            if "selection" in question_types:
                selection_question = create_selection_question(first_sent, level)
                sentence_data["selection_question"] = selection_question
            if "labeling" in question_types:
                labeling_question = create_labeling_question(first_sent, level)
                sentence_data["labeling_question"] = labeling_question
            if "fill_in_blank" in question_types:
                fill_in_blank_question = create_fill_in_blank(first_sent, level)
                sentence_data["fill_in_blank_question"] = fill_in_blank_question
            if "multiple_choice" in question_types:
                multiple_choice_question = create_multiple_choice(first_sent, level)
                sentence_data["multiple_choice_question"] = multiple_choice_question
            if "arrange" in question_types:
                arrange_question = create_arrange_question(first_sent, level)
                sentence_data["arrange_question"] = arrange_question
            if not any([sentence_data.get(qt + "_question") for qt in question_types]):
                print("Debug: No valid questions generated, skipping.")
                continue
            validated_sentences.append(sentence_data)
            print(f"Debug: Added valid sentence: '{first_sent}'")
        except Exception as e:
            print(f"Error generating sentence: {str(e)}")
            continue

    if not validated_sentences:
        print(f"Warning: No valid sentences generated after {attempt_count} attempts.")

    print("\nError Statistics:")
    print(f"Duplicate sentence = {error_counts['duplicate_sentence']} Case")
    print(f"Sentence CEFR level mismatch = {error_counts['cefr_mismatch']} Case")
    print(f"No valid verb found = {error_counts['no_valid_verb']} Case")
    print(f"No valid constituents = {error_counts['no_valid_constituents']} Case")
    print(f"Insufficient constituents = {error_counts['insufficient_constituents']} Case")
    print(f"Multiple clauses = {error_counts['multiple_clauses']} Case")
    print(f"Passive voice = {error_counts['passive_voice']} Case")
    print(f"Duplicate selection sentences = {error_counts['duplicate_selection']} Case")
    print(f"Invalid incorrect sentences = {error_counts['invalid_incorrect_sentence']} Case")
    print(f"Invalid fill-in-the-blank = {error_counts['invalid_fill_in_blank']} Case")
    print(f"Invalid multiple choice = {error_counts['invalid_multiple_choice']} Case")
    print(f"Invalid arrange questions = {error_counts['invalid_arrange']} Case")
    print(f"No full-stop in sentence = {error_counts['no_full_stop']} Case")

    return validated_sentences

# Evaluation functions
def evaluate_selection(selected: str, correct: str) -> Dict:
    selected = clean_sentence(selected)
    if selected is None:
        return {
            "is_correct": False,
            "score": 0.0,
            "message": "Invalid selected sentence: no full-stop found."
        }
    correct = clean_sentence(correct)
    if correct is None:
        return {
            "is_correct": False,
            "score": 0.0,
            "message": "Invalid correct sentence: no full-stop found."
        }
    is_correct = selected.strip() == correct.strip()
    score = 1.0 if is_correct else 0.0
    return {
        "is_correct": is_correct,
        "score": score,
        "message": "Selection is correct." if is_correct else "Selection is incorrect."
    }

def evaluate_labeling(sentence: str, user_labels: Dict[str, str]) -> Dict:
    global SPACY_MODEL
    cleaned_sentence = clean_sentence(sentence)
    if cleaned_sentence is None:
        return {
            "is_correct": False,
            "score": 0.0,
            "message": "Invalid sentence: no full-stop found."
        }
    doc = SPACY_MODEL(cleaned_sentence)
    correct_labels = {
        "subject": None,
        "verb": None,
        "object": None,
        "adjective": None,
        "adverb": None,
        "prepositional phrase": None
    }
    prep_obj = None
    main_verb = None
    for token in doc:
        if token.dep_ == "nsubj" and not correct_labels["subject"]:
            correct_labels["subject"] = token.text
            for child in token.children:
                if child.dep_ in ["det", "compound"]:
                    correct_labels["subject"] = f"{child.text} {correct_labels['subject']}"
        if token.dep_ == "ROOT" and token.pos_ in ["VERB", "AUX"]:
            if token.pos_ == "AUX" and not main_verb:
                for child in token.children:
                    if child.pos_ == "VERB" and child.dep_ in ["xcomp", "ccomp", "advcl"]:
                        main_verb = child.text
                        break
            if not main_verb:
                main_verb = token.text
        if token.dep_ == "dobj" and not correct_labels["object"]:
            obj = token.text
            for child in token.children:
                if child.dep_ in ["det", "amod", "compound"]:
                    obj = f"{child.text} {obj}"
            correct_labels["object"] = obj
        if token.dep_ in ["amod", "acomp"] and token.pos_ == "ADJ" and not correct_labels["adjective"]:
            correct_labels["adjective"] = token.text
        if token.dep_ == "advmod" and token.pos_ == "ADV" and not correct_labels["prepositional phrase"]:
            for child in token.head.children:
                if child.dep_ == "prep" and child.text in ["around", "about"]:
                    correct_labels["prepositional phrase"] = f"{child.text}"
                    for gc in child.children:
                        if gc.dep_ == "pobj":
                            correct_labels["prepositional phrase"] += f" {gc.text}"
                            break
                    print(f"Debug: Relabeled '{token.text}' as prepositional phrase")
                    break
            else:
                correct_labels["adverb"] = token.text
        if token.dep_ == "prep" and not correct_labels["prepositional phrase"]:
            for child in token.children:
                if child.dep_ == "pobj":
                    prep_obj = child.text
                    for grandchild in child.children:
                        if grandchild.dep_ in ["det", "amod"]:
                            prep_obj = f"{grandchild.text} {prep_obj}"
                    correct_labels["prepositional phrase"] = f"{token.text} {prep_obj}"
                    break
    correct_labels["verb"] = main_verb
    if correct_labels["verb"]:
        for contraction, expansion in {
            "'ve": "have",
            "'s": "is",
            "'re": "are",
            "'m": "am",
            "'ll": "will",
            "'d": "had"
        }.items():
            if contraction in correct_labels["verb"].lower():
                correct_labels["verb"] = expansion
    valid_correct_labels = {k: v for k, v in correct_labels.items() if v is not None}
    if not valid_correct_labels:
        return {
            "is_correct": False,
            "score": 0.0,
            "message": "No valid constituents found in sentence."
        }
    score_per_label = 1.0 / len(valid_correct_labels)
    is_correct = all(
        user_labels.get(key, "").strip().lower() == value.strip().lower()
        for key, value in valid_correct_labels.items()
    )
    score = sum(
        score_per_label
        for key, value in valid_correct_labels.items()
        if user_labels.get(key, "").strip().lower() == value.strip().lower()
    )
    return {
        "is_correct": is_correct,
        "score": round(score, 2),
        "message": "Labeling is correct." if is_correct else "Labeling is incorrect."
    }

def evaluate_fill_in_blank(user_answer: str, correct_answer: str) -> Dict:
    is_correct = user_answer.strip().lower() == correct_answer.strip().lower()
    score = 1.0 if is_correct else 0.0
    return {
        "is_correct": is_correct,
        "score": score,
        "message": "Answer is correct." if is_correct else "Answer is incorrect."
    }

def evaluate_multiple_choice(selected_option: str, correct_answer: str) -> Dict:
    is_correct = selected_option.strip().lower() == correct_answer.strip().lower()
    score = 1.0 if is_correct else 0.0
    return {
        "is_correct": is_correct,
        "score": score,
        "message": "Selection is correct." if is_correct else "Selection is incorrect."
    }

def evaluate_arrange(user_arrangement: List[str], correct_sentence: str) -> Dict:
    user_sentence = " ".join(user_arrangement).strip()
    correct_sentence = clean_sentence(correct_sentence)
    if correct_sentence is None:
        return {
            "is_correct": False,
            "score": 0.0,
            "message": "Invalid correct sentence: no full-stop found."
        }
    is_correct = user_sentence.lower() == correct_sentence.lower()
    score = 1.0 if is_correct else 0.0
    return {
        "is_correct": is_correct,
        "score": score,
        "message": "Arrangement is correct." if is_correct else "Arrangement is incorrect."
    }

# API endpoints
@app.post("/generate_sentences", response_model=SentenceResponse)
async def generate_sentences(request: Request, sentence_request: SentenceRequest):
    start_time = time.time()
    try:
        level = sentence_request.level
        num_sentences = sentence_request.num_sentences
        user_id = sentence_request.user_id
        topic = sentence_request.topic
        question_types = sentence_request.question_types
        valid_question_types = ["selection", "labeling", "fill_in_blank", "multiple_choice", "arrange"]
        if not all(qt in valid_question_types for qt in question_types):
            raise HTTPException(status_code=400, detail=f"Invalid question types. Must be subset of: {valid_question_types}")
        if level not in ["A1", "A2", "B1", "B2", "C1", "C2"]:
            raise HTTPException(status_code=400, detail="Invalid level. Must be one of: A1, A2, B1, B2, C1, C2")
        if num_sentences < 1 or num_sentences > 10:
            raise HTTPException(status_code=400, detail="Number of sentences must be between 1 and 10")
        sentences = generate_sentences_internal(level, num_sentences, user_id, topic, question_types)
        processing_time = time.time() - start_time
        current_time = datetime.datetime.now().isoformat()
        if not sentences:
            print("Debug: No sentences generated, returning empty response")
            return {
                "success": False,
                "sentences": [],
                "message": "Failed to generate sentences. Please try again.",
                "processing_time": processing_time,
                "timestamp": current_time
            }
        response = {
            "success": True,
            "sentences": sentences,
            "message": f"Generated {len(sentences)} sentences for level {level}" +
                      (f" on topic '{topic}'" if topic else "") +
                      f" with question types {question_types}",
            "processing_time": processing_time,
            "timestamp": current_time
        }
        print(f"Debug: Sending response: {response}")
        return response
    except HTTPException:
        raise
    except Exception as e:
        processing_time = time.time() - start_time
        current_time = datetime.datetime.now().isoformat()
        print(f"API error: {str(e)}")
        return {
            "success": False,
            "sentences": [],
            "message": f"Internal server error: {str(e)}",
            "processing_time": processing_time,
            "timestamp": current_time
        }

@app.post("/evaluate_selection", response_model=SelectionResponse)
async def evaluate_selection(request: Request, selection_request: SelectionRequest):
    start_time = time.time()
    try:
        result = evaluate_selection(
            selection_request.selected_sentence,
            selection_request.correct_sentence
        )
        processing_time = time.time() - start_time
        current_time = datetime.datetime.now().isoformat()
        return {
            "success": True,
            "is_correct": result["is_correct"],
            "score": result["score"],
            "message": result["message"],
            "processing_time": processing_time,
            "timestamp": current_time
        }
    except Exception as e:
        processing_time = time.time() - start_time
        current_time = datetime.datetime.now().isoformat()
        return {
            "success": False,
            "is_correct": False,
            "score": 0.0,
            "message": f"Error evaluating selection: {str(e)}",
            "processing_time": processing_time,
            "timestamp": current_time
        }

def run_server(port: int = 8000, max_attempts: int = 5):
    import socket
    for attempt in range(max_attempts):
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                s.bind(("0.0.0.0", port))
            config = uvicorn.Config(
                app,
                host="0.0.0.0",
                port=port,
                log_level="info",
                timeout_keep_alive=30
            )
            server = uvicorn.Server(config)
            print(f"Starting server on port {port}...")
            asyncio.run(server.serve())
            break
        except OSError as e:
            if "address already in use" in str(e).lower() and attempt < max_attempts - 1:
                print(f"Port {port} in use, trying port {port + 1}...")
                port += 1
                time.sleep(1)
            else:
                print(f"Failed to start server: {str(e)}")
                raise
        except Exception as e:
            print(f"Server error: {str(e)}")
            raise

BASE_MODEL = "unsloth/mistral-7b-v0.3-bnb-4bit"
ADAPTER = "Justin73/grammar-correction-model-combined-mistral"
CORRECTION_MODEL = None
CORRECTION_TOKENIZER = None

def load_correction_model():
    global CORRECTION_MODEL, CORRECTION_TOKENIZER
    try:
        print("Loading correction model...")
        CORRECTION_MODEL, CORRECTION_TOKENIZER = FastLanguageModel.from_pretrained(
            BASE_MODEL,
            max_seq_length=2048,
            load_in_4bit=True,
        )
        CORRECTION_MODEL = PeftModel.from_pretrained(CORRECTION_MODEL, ADAPTER)
        CORRECTION_TOKENIZER.pad_token = CORRECTION_TOKENIZER.eos_token
        CORRECTION_MODEL = FastLanguageModel.for_inference(CORRECTION_MODEL)
        print("Correction model loaded!")
        return True
    except Exception as e:
        print(f"Correction model loading failed: {str(e)}")
        return False

@app.post("/grammar_correction", response_model=GrammarCorrectionResponse)
async def grammar_correction(request: GrammarCorrectionRequest):
    if CORRECTION_MODEL is None or CORRECTION_TOKENIZER is None:
        raise HTTPException(status_code=503, detail="Correction model not loaded")

    try:
        prompt = f"""### Instruction:
Correct all the grammatical errors in the following sentence.

### Input:
{request.sentence}

### Response:
"""

        inputs = CORRECTION_TOKENIZER(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        outputs = CORRECTION_MODEL.generate(
            **inputs,
            max_new_tokens=384,
            temperature=0.1,
            pad_token_id=CORRECTION_TOKENIZER.pad_token_id
        )

        full_output = CORRECTION_TOKENIZER.decode(outputs[0], skip_special_tokens=True)

        # Parse corrected text
        corrected_match = re.search(r'Corrected text:(.*?)(\n|$)', full_output, re.DOTALL)
        corrected_text = corrected_match.group(1).strip() if corrected_match else request.sentence

        # Parse corrections
        corrections = []
        corrections_section = re.search(r'Corrections:(.*?)(\n\n|$)', full_output, re.DOTALL)
        if corrections_section:
            corrections_text = corrections_section.group(1).strip()
            corrections = [
                line.strip()
                for line in corrections_text.split('\n')
                if line.strip() and re.match(r'^\d+\.', line)
            ]

        return {
            "success": True,
            "user_id": request.user_id,
            "original": request.sentence,
            "corrected": corrected_text,
            "corrections": corrections
        }

    except Exception as e:
        return {
            "success": False,
            "user_id": request.user_id,
            "original": request.sentence,
            "corrected": request.sentence,
            "corrections": []
        }
        
def load_all_models():
    """Load all models for both services"""
    print("Loading all models for both services...")
    load_generator_models()
    load_correction_model()
    print("All models loaded successfully!")
    
def start_ngrok(port: int) -> Optional[str]:
    try:
        ngrok.kill()
        ngrok.set_auth_token("2oFl98p58vTuHbdQu5m2t1exmDW_5MXsrmeuZtv9J5hTscGzb")
        public_url = ngrok.connect(port, bind_tls=True).public_url
        print(f"Ngrok tunnel created: {public_url}")
        return public_url
    except Exception as e:
        print(f"Ngrok error: {str(e)}")
        print("Continuing without ngrok - you'll need to access the API locally")
        return None

def start_application():
    try:
        load_all_models()
    except Exception as e:
        print(f"Failed to load models: {str(e)}")
        return
    port = 8000
    max_attempts = 5
    attempt = 0
    while attempt < max_attempts:
        try:
            server_thread = threading.Thread(target=run_server, args=(port,))
            server_thread.daemon = True
            server_thread.start()
            time.sleep(2)
            public_url = start_ngrok(port)
            if public_url:
                print(f"\nAPI Endpoints:")
                print(f"- Sentence Generation: {public_url}/generate_sentences")
                print(f"- Grammar Correction: {public_url}/grammar_correction")
                print(f"- Swagger UI: {public_url}/docs")
            else:
                print(f"\nAPI is running locally at:")
                print(f"- http://localhost:{port}/generate_sentences")
                print(f"- http://localhost:{port}/grammar_correction")
                print(f"- http://localhost:{port}/docs")
            while True:
                time.sleep(1)
        except KeyboardInterrupt:
            print("\nShutting down server...")
            ngrok.kill()
            import os
            os._exit(0)
        except Exception as e:
            print(f"Application error: {str(e)}")
            ngrok.kill()
            attempt += 1
            port += 1
            time.sleep(2)
            if attempt < max_attempts:
                print(f"Retrying with port {port}...")

if __name__ == "__main__":
    start_application()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading all models for both services...
Loading all models...
==((====))==  Unsloth 2025.5.1: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.988 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.5.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Finetuned generator model 'Mr-FineTuner/With_synthetic_Dataset_llama-1epoch' loaded successfully!
Loading plain distractor model...
==((====))==  Unsloth 2025.5.1: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.988 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Plain distractor model 'unsloth/llama-3-8b-Instruct-bnb-4bit' loaded successfully!
Loading validator model...
Validator models loaded successfully!
Loading spaCy model...
spaCy model loaded successfully!
Loading correction model...
==((====))==  Unsloth 2025.5.1: Fast Mistral patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.988 GB. Platform: Lin

INFO:     Started server process [13375]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Correction model loaded!
All models loaded successfully!
Starting server on port 8000...
Ngrok tunnel created: https://0416-103-119-147-234.ngrok-free.app

API Endpoints:
- Sentence Generation: https://0416-103-119-147-234.ngrok-free.app/generate_sentences
- Grammar Correction: https://0416-103-119-147-234.ngrok-free.app/grammar_correction
- Swagger UI: https://0416-103-119-147-234.ngrok-free.app/docs


In [ ]:
from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel
import uvicorn
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
import joblib
import numpy as np
import re
import random
from huggingface_hub import snapshot_download
import torch
import spacy
from pyngrok import ngrok
import nest_asyncio
import threading
import asyncio
import time
from typing import List, Dict, Optional
from transformers import StoppingCriteria
import datetime
import os

# Allow nested event loops for Colab
nest_asyncio.apply()

# Define FastAPI app
app = FastAPI()

# Global variables for loaded models and constants
GENERATOR_MODEL = None
GENERATOR_TOKENIZER = None
DISTRACTOR_MODEL = None
DISTRACTOR_TOKENIZER = None
EVALUATOR_MODEL = None
EMBEDDER_MODEL = None
SPACY_MODEL = None
MAX_SEQ_LENGTH = 2048

# Define request models
class SentenceRequest(BaseModel):
    level: str
    num_sentences: int = 1
    user_id: int
    topic: Optional[str] = None
    question_types: Optional[List[str]] = ["selection", "labeling", "fill_in_blank", "multiple_choice", "arrange"]

class SelectionRequest(BaseModel):
    selected_sentence: str
    sentences: str  # Changed to str to match the | delimiter
    correct_sentence: str
    target_level: str
    user_id: int

class LabelingRequest(BaseModel):
    sentence: str
    user_labels: Dict[str, str]
    target_level: str
    user_id: int

class FillInBlankRequest(BaseModel):
    sentence: str
    user_answer: str
    correct_answer: str
    target_level: str
    user_id: int

class MultipleChoiceRequest(BaseModel):
    selected_option: str
    options: List[str]
    correct_answer: str
    target_level: str
    user_id: int

class ArrangeRequest(BaseModel):
    sentence: str
    user_arrangement: List[str]
    correct_sentence: str
    target_level: str
    user_id: int

# Define response models
class SentenceResponse(BaseModel):
    success: bool
    sentences: List[Dict]
    message: str
    processing_time: float
    timestamp: str

class SelectionResponse(BaseModel):
    success: bool
    is_correct: bool
    score: float
    message: str
    processing_time: float
    timestamp: str

class LabelingResponse(BaseModel):
    success: bool
    is_correct: bool
    score: float
    message: str
    processing_time: float
    timestamp: str

class FillInBlankResponse(BaseModel):
    success: bool
    is_correct: bool
    score: float
    message: str
    processing_time: float
    timestamp: str

class MultipleChoiceResponse(BaseModel):
    success: bool
    is_correct: bool
    score: float
    message: str
    processing_time: float
    timestamp: str

class ArrangeResponse(BaseModel):
    success: bool
    is_correct: bool
    score: float
    message: str
    processing_time: float
    timestamp: str

class StopOnTokens(StoppingCriteria):
    def __init__(self, tokenizer, stop_strings):
        self.tokenizer = tokenizer
        self.stop_strings = stop_strings
        self.stop_ids = [tokenizer.encode(s, add_special_tokens=False) for s in stop_strings]

    def __call__(self, input_ids, scores, **kwargs):
        for stop_id in self.stop_ids:
            if len(input_ids[0]) >= len(stop_id):
                if all(input_ids[0][-len(stop_id):] == stop_id):
                    return True
        return False

# Model loading function
def load_models():
    global GENERATOR_MODEL, GENERATOR_TOKENIZER, DISTRACTOR_MODEL, DISTRACTOR_TOKENIZER, EVALUATOR_MODEL, EMBEDDER_MODEL, SPACY_MODEL
    print("Loading all models...")
    dtype = None
    load_in_4bit = True
    # generator_model_name = "Mr-FineTuner/Test_02_llama_1epoch_trainPercen_myValidator_fix"
    generator_model_name = "Mr-FineTuner/With_synthetic_Dataset_llama-1epoch"
    try:
        GENERATOR_MODEL, GENERATOR_TOKENIZER = FastLanguageModel.from_pretrained(
            model_name=generator_model_name,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=dtype,
            load_in_4bit=load_in_4bit,
        )
        if GENERATOR_TOKENIZER.pad_token is None:
            GENERATOR_TOKENIZER.pad_token = GENERATOR_TOKENIZER.eos_token
        GENERATOR_MODEL = FastLanguageModel.for_inference(GENERATOR_MODEL)
        print(f"Finetuned generator model '{generator_model_name}' loaded successfully!")
    except Exception as e:
        print(f"Error loading finetuned generator model: {str(e)}")
        raise
    print("Loading plain distractor model...")
    distractor_model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
    try:
        DISTRACTOR_MODEL, DISTRACTOR_TOKENIZER = FastLanguageModel.from_pretrained(
            model_name=distractor_model_name,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=dtype,
            load_in_4bit=load_in_4bit,
        )
        if DISTRACTOR_TOKENIZER.pad_token is None:
            DISTRACTOR_TOKENIZER.pad_token = DISTRACTOR_TOKENIZER.eos_token
        DISTRACTOR_MODEL = FastLanguageModel.for_inference(DISTRACTOR_MODEL)
        print(f"Plain distractor model '{distractor_model_name}' loaded successfully!")
    except Exception as e:
        print(f"Error loading plain distractor model: {str(e)}")
        raise
    print("Loading validator model...")
    try:
        eval_model_path = snapshot_download(
            repo_id="Mr-FineTuner/Skripsi_validator_best_model",
            allow_patterns=["model_mlp.joblib"],
            local_dir="validator_model"
        )
        classifier_path = os.path.join(eval_model_path, "model_mlp.joblib")
        if not os.path.exists(classifier_path):
            raise FileNotFoundError(f"model_mlp.joblib not found in {eval_model_path}")
        EVALUATOR_MODEL = joblib.load(classifier_path)
        EMBEDDER_MODEL = SentenceTransformer('BAAI/bge-base-en-v1.5')
        print("Validator models loaded successfully!")
    except Exception as e:
        print(f"Error loading validator models: {str(e)}")
        raise
    print("Loading spaCy model...")
    try:
        SPACY_MODEL = spacy.load("en_core_web_sm")
        print("spaCy model loaded successfully!")
    except Exception as e:
        print(f"Error loading spaCy model: {str(e)}")
        raise

# Sentence generation logic
def generate_sentences_internal(level: str, num_sentences: int, user_id: int, topic: Optional[str] = None, question_types: List[str] = ["selection", "labeling", "fill_in_blank", "multiple_choice", "arrange"]) -> List[Dict]:
    global GENERATOR_MODEL, GENERATOR_TOKENIZER, DISTRACTOR_MODEL, DISTRACTOR_TOKENIZER, EVALUATOR_MODEL, EMBEDDER_MODEL, SPACY_MODEL
    if None in [GENERATOR_MODEL, GENERATOR_TOKENIZER, DISTRACTOR_MODEL, DISTRACTOR_TOKENIZER, EVALUATOR_MODEL, EMBEDDER_MODEL, SPACY_MODEL]:
        raise ValueError("One or more models failed to load")

    # Contraction mapping for expansion
    CONTRACTION_MAP = {
        "'ve": "have",
        "'s": "is",
        "'re": "are",
        "'m": "am",
        "'ll": "will",
        "'d": "had"
    }

    error_counts = {
        "duplicate_sentence": 0,
        "cefr_mismatch": 0,
        "no_valid_verb": 0,
        "no_valid_constituents": 0,
        "insufficient_constituents": 0,
        "multiple_clauses": 0,
        "passive_voice": 0,
        "output_pattern_removed": 0,
        "duplicate_selection": 0,
        "invalid_incorrect_sentence": 0,
        "invalid_fill_in_blank": 0,
        "invalid_multiple_choice": 0,
        "invalid_arrange": 0,
        "no_full_stop": 0
    }

    def predict_sentiment(text: str) -> int:
        embedding = EMBEDDER_MODEL.encode([text])
        probs = EVALUATOR_MODEL.predict_proba(embedding)[0]
        prediction = np.argmax(probs) + 1
        return max(1, min(6, int(prediction)))

    cefr_to_score = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
    score_to_cefr = {1: 'A1', 2: 'A2', 3: 'B1', 4: 'B2', 5: 'C1', 6: 'C2'}

    def clean_sentence(text: str) -> Optional[str]:
        cleaned = re.sub(r'\n+#+\s*Output\s*:', '', text, flags=re.IGNORECASE)
        cleaned = re.sub(r'\n+\s*###\s*Output\s*:', '', cleaned, flags=re.IGNORECASE)
        cleaned = cleaned.strip()
        match = re.match(r'^(.*?)\.', cleaned)
        if match:
            cleaned = match.group(1) + '.'
        else:
            print("Debug: No full-stop found in sentence, rejecting.")
            return None
        cleaned = cleaned.strip()
        if not cleaned:
            print("Debug: Cleaned sentence is empty, rejecting.")
            return None
        return cleaned

    def is_article_correct(sent: str, orig_doc) -> bool:
        doc = SPACY_MODEL(sent)
        for i, token in enumerate(doc):
            if token.dep_ == "nsubj" and i > 0:
                prev = doc[i-1].text.lower()
                if prev in ["a", "an"]:
                    first_letter = orig_doc[i].text[0].lower()
                    is_vowel = first_letter in 'aeiou'
                    if (prev == "a" and is_vowel) or (prev == "an" and not is_vowel):
                        return False
                elif prev == "the":
                    return False
            elif token.dep_ == "nsubj" and i == 0:
                if token.tag_ in ["NN", "NNP"]:
                    return False
        return True

    def is_subject_verb_agreement_correct(sent: str) -> bool:
        doc = SPACY_MODEL(sent)
        subject = None
        verb = None
        for token in doc:
            if token.dep_ == "nsubj":
                subject = token
            if token.dep_ == "ROOT" and token.pos_ in ["VERB", "AUX"]:
                verb = token
        if subject and verb:
            is_subject_plural = subject.tag_ in ["NNS", "NNPS"]
            is_verb_singular = verb.text.lower() in ["is", "was", "has"]
            return not (is_subject_plural != is_verb_singular)
        return True

    def create_selection_question(sentence: str, level: str) -> Dict:
        error_types = {
            'A1': ["spelling", "article", "number_agreement"],
            'A2': ["spelling", "article", "tense"],
            'B1': ["tense", "preposition", "word_choice"],
            'B2': ["tense", "preposition", "agreement"],
            'C1': ["complex_tense", "idiom", "word_form"],
            'C2': ["complex_tense", "idiom", "subjunctive"]
        }
        available_errors = error_types.get(level, error_types['B1'])
        if len(available_errors) < 2:
            available_errors = available_errors + ["spelling"]
        errors = random.sample(available_errors, 2)

        doc = SPACY_MODEL(sentence)
        words = [token.text for token in doc]

        def introduce_error(sent: str, error_type: str) -> str:
            doc = SPACY_MODEL(sent)
            words = [token.text for token in doc]
            modified = sent
            if error_type == "spelling":
                for i, word in enumerate(words):
                    if len(word) > 3 and word.isalpha():
                        if len(word) > 4:
                            words[i] = word[:-2] + random.choice('ae') + random.choice('mn')
                        else:
                            words[i] = word[:-1] + random.choice('ae')
                        modified = " ".join(words).strip()
                        if SPACY_MODEL(words[i])[0].is_oov:
                            break
                else:
                    modified = sent
            elif error_type == "article":
                for i, token in enumerate(doc):
                    if token.dep_ == "nsubj" and i > 0 and words[i-1].lower() in ["a", "an"]:
                        words[i-1] = ""
                        modified = " ".join(words).strip()
                        break
                    elif token.dep_ == "nsubj":
                        first_letter = token.text[0].lower()
                        is_vowel = first_letter in 'aeiou'
                        words[i] = f"{'a' if is_vowel else 'an'} {words[i]}"
                        modified = " ".join(words).strip()
                        break
                else:
                    modified = sent
            elif error_type == "number_agreement":
                for i, token in enumerate(doc):
                    if token.dep_ == "nsubj":
                        is_plural = token.tag_ in ["NNS", "NNPS"]
                        if is_plural:
                            words[i] = SPACY_MODEL(token.lemma_)[0].text
                        else:
                            words[i] = token.text + "s"
                        for j, t in enumerate(doc):
                            if t.dep_ == "ROOT" and t.text.lower() in ["is", "are"]:
                                words[j] = "are" if t.text.lower() == "is" else "is"
                                break
                        modified = " ".join(words).strip()
                        break
                else:
                    modified = sent
            elif error_type == "tense":
                for i, token in enumerate(doc):
                    if token.pos_ == "VERB" and token.text.lower() not in ["be", "can"]:
                        words[i] = SPACY_MODEL(token.lemma_)[0].text + "ed"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "preposition":
                for i, token in enumerate(doc):
                    if token.dep_ == "prep":
                        words[i] = "to" if token.text != "to" else "in"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "word_choice":
                for i, token in enumerate(doc):
                    if token.dep_ == "dobj":
                        words[i] = "banana" if token.text.lower() != "banana" else "orange"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "agreement":
                for i, token in enumerate(doc):
                    if token.dep_ == "nsubj" and token.text.lower() == "i":
                        words[i] = "We"
                        for v in doc:
                            if v.pos_ == "VERB" and v.text.lower() in ["like", "eat"]:
                                words[v.i] = v.text + "s"
                                break
                        modified = " ".join(words).strip()
                        break
            elif error_type == "complex_tense":
                for i, token in enumerate(doc):
                    if token.pos_ == "VERB":
                        words[i] = f"had {token.lemma_}ed"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "idiom":
                for i, token in enumerate(doc):
                    if token.dep_ == "dobj":
                        words[i] = "moon"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "word_form":
                for i, token in enumerate(doc):
                    if token.pos_ == "VERB":
                        words[i] = token.lemma_ + "ing"
                        modified = " ".join(words).strip()
                        break
            elif error_type == "subjunctive":
                for i, token in enumerate(doc):
                    if token.pos_ == "VERB":
                        words[i] = token.lemma_
                        modified = " ".join(words).strip()
                        break
            modified = clean_sentence(modified)
            if modified is None or modified.strip() == sent.strip():
                for i, word in enumerate(words):
                    if len(word) > 3 and word.isalpha():
                        words[i] = word[:-1] + random.choice('ae')
                        modified = clean_sentence(" ".join(words).strip())
                        break
            return modified if modified else sent

        incorrect1 = introduce_error(sentence, errors[0])
        incorrect2 = introduce_error(sentence, errors[1])
        max_error_retries = 3
        error_retry_count = 0
        while (incorrect2 == incorrect1 or incorrect2 == sentence) and error_retry_count < max_error_retries:
            new_error = random.choice([e for e in available_errors if e != errors[0]])
            incorrect2 = introduce_error(sentence, new_error)
            errors[1] = new_error
            error_retry_count += 1
        if incorrect2 == incorrect1 or incorrect2 == sentence:
            for i, word in enumerate(words):
                if len(word) > 3 and word.isalpha():
                    words[i] = word[:-1] + 'u'
                    incorrect2 = clean_sentence(" ".join(words).strip())
                    errors[1] = "spelling"
                    break
        sentences = [sentence, incorrect1, incorrect2]
        for i, s in enumerate(sentences):
            if s != sentence:
                is_valid = True
                try:
                    if errors[i-1] == "article" and is_article_correct(s, doc):
                        is_valid = False
                    elif errors[i-1] == "number_agreement" and is_subject_verb_agreement_correct(s):
                        is_valid = False
                    elif errors[i-1] == "spelling":
                        s_doc = SPACY_MODEL(s)
                        for token, orig_token in zip(s_doc, doc):
                            if token.text != orig_token.text and not token.is_oov:
                                is_valid = False
                                break
                except Exception as e:
                    print(f"Debug: Validation error for '{s}': {str(e)}")
                    is_valid = False
                if not is_valid:
                    print(f"Debug: Invalid incorrect sentence '{s}' for error '{errors[i-1]}'")
                    error_counts["invalid_incorrect_sentence"] += 1
                    new_error = random.choice([e for e in available_errors if e != errors[i-1]])
                    sentences[i] = introduce_error(sentence, new_error)
                    errors[i-1] = new_error
        if len(set(sentences)) != 3:
            print(f"Debug: Duplicate sentences detected in selection question: {sentences}")
            error_counts["duplicate_selection"] += 1
            incorrect2 = introduce_error(sentence, random.choice(available_errors))
            sentences = [sentence, incorrect1, incorrect2]
            if len(set(sentences)) != 3:
                for i, word in enumerate(words):
                    if len(word) > 3 and word.isalpha():
                        words[i] = word[:-1] + 'u'
                        incorrect2 = clean_sentence(" ".join(words).strip())
                        sentences = [sentence, incorrect1, incorrect2]
                        break
        random.shuffle(sentences)
        return {
            "question_text": "Choose the correct sentence.",
            "sentences": "|".join(sentences),  # Use | as delimiter
            "correct_sentence": sentence
        }

    def create_fill_in_blank(sentence: str, level: str) -> Optional[Dict]:
        doc = SPACY_MODEL(sentence)
        verb = None
        verb_index = None
        verb_lemma = None
        for i, token in enumerate(doc):
            if token.pos_ in ["VERB", "AUX"] and token.dep_ in ["ROOT", "cop"]:
                if token.text.lower() not in ["to"]:
                    verb = token.text
                    verb_index = i
                    verb_lemma = token.lemma_.lower()
                    break
        if not verb or not verb_lemma:
            print("Debug: No valid verb for fill-in-the-blank, skipping.")
            error_counts["invalid_fill_in_blank"] += 1
            return None
        words = [token.text for token in doc]
        words[verb_index] = "__"
        blanked_sentence = " ".join(words).strip()
        question_text = f"{blanked_sentence[:-1]} ({verb_lemma})."
        return {
            "question_text": question_text,
            "correct_answer": verb
        }

    def generate_contextual_distractors(target_verb: str, sentence: str, num_distractors: int = 3) -> List[str]:
        print(f"Debug: Starting distractor generation for verb '{target_verb}' in sentence '{sentence}'")
        target_doc = SPACY_MODEL(target_verb)
        target_tense = None
        for token in target_doc:
            if token.pos_ == "VERB":
                target_tense = token.tag_
                break
        print(f"Debug: Target verb tense tag: {target_tense if target_tense else 'None'}")
        tense_map = {
            'VB': 'base form',
            'VBD': 'past tense',
            'VBG': 'gerund/present participle',
            'VBN': 'past participle',
            'VBP': 'present tense (non-3rd person singular)',
            'VBZ': 'present tense (3rd person singular)'
        }
        tense_desc = tense_map.get(target_tense, 'base form') if target_tense else 'base form'
        print(f"Debug: Tense description for prompt: {tense_desc}")
        total_distractors_to_generate = 6
        prompt = (
            f"Generate exactly {total_distractors_to_generate} English verbs that are distinct from '{target_verb}', "
            f"semantically related, and appropriate for CEFR level {level}. "
            f"The verbs must fit the context of the sentence '{sentence}' "
            f"and be in the {tense_desc} (e.g., '{target_verb}'). "
            f"Return only the verbs as a comma-separated list, with no additional text or non-verbs."
        )
        print(f"Debug: LLM prompt:\n{prompt}")
        try:
            input_data = DISTRACTOR_TOKENIZER(
                prompt,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).to("cuda" if torch.cuda.is_available() else "cpu")
            print(f"Debug: Tokenized input data: {input_data}")
            outputs = DISTRACTOR_MODEL.generate(
                input_ids=input_data["input_ids"],
                attention_mask=input_data["attention_mask"],
                max_new_tokens=50,
                pad_token_id=DISTRACTOR_TOKENIZER.pad_token_id,
                do_sample=True,
                temperature=0.6,
                top_k=50,
                top_p=0.85,
            )
            print(f"Debug: Raw LLM output: {outputs}")
            generated_text = DISTRACTOR_TOKENIZER.decode(outputs[0], skip_special_tokens=True)
            print(f"Debug: Decoded LLM output: '{generated_text}'")
            response_start = generated_text.find("<|start_header_id|>assistant<|end_header_id>") + len("<|start_header_id|>assistant<|end_header_id>")
            response_text = generated_text[response_start:].strip()
            print(f"Debug: Extracted response text: '{response_text}'")
            distractors = [v.strip() for v in response_text.split(",") if v.strip()]
            print(f"Debug: Parsed distractors: {distractors}")
            primary_distractors = []
            backup_distractors = []
            for verb in distractors:
                if not (len(verb.split()) == 1 and re.match(r'^[a-zA-Z]+$', verb)):
                    print(f"Debug: Invalid distractor '{verb}' - not a single alphabetic word")
                    continue
                if verb.lower() == target_verb.lower():
                    print(f"Debug: Invalid distractor '{verb}' - matches target verb")
                    continue
                doc = SPACY_MODEL(verb)
                is_verb = any(token.pos_ == "VERB" for token in doc)
                print(f"Debug: Validating '{verb}' - is_verb: {is_verb}")
                if not is_verb:
                    print(f"Debug: Invalid distractor '{verb}' - not recognized as a verb")
                    continue
                if len(primary_distractors) < num_distractors:
                    primary_distractors.append(verb)
                elif len(backup_distractors) < num_distractors:
                    backup_distractors.append(verb)
                if len(primary_distractors) >= num_distractors and len(backup_distractors) >= num_distractors:
                    break
            print(f"Debug: Primary distractors: {primary_distractors}")
            print(f"Debug: Backup distractors: {backup_distractors}")
            selected_distractors = primary_distractors[:num_distractors]
            if len(selected_distractors) < num_distractors:
                print(f"Debug: Insufficient primary distractors ({len(selected_distractors)}/{num_distractors}). Using backups.")
                needed = num_distractors - len(selected_distractors)
                valid_backups = [v for v in backup_distractors if v not in selected_distractors]
                selected_distractors.extend(valid_backups[:needed])
                print(f"Debug: Added backups: {valid_backups[:needed]}")
            if len(selected_distractors) < num_distractors:
                print(f"Debug: Insufficient distractors after backups ({len(selected_distractors)}/{num_distractors}). Using fallbacks.")
                fallback_verbs = ["walk", "talk", "look", "move", "say"]
                fallback_verbs = [v for v in fallback_verbs if v.lower() != target_verb.lower() and v not in selected_distractors]
                selected_distractors.extend(fallback_verbs[:num_distractors - len(selected_distractors)])
                print(f"Debug: Added fallbacks: {selected_distractors[len(selected_distractors) - (num_distractors - len(selected_distractors)):]}")
            print(f"Debug: Final distractors: {selected_distractors[:num_distractors]}")
            return selected_distractors[:num_distractors]
        except Exception as e:
            print(f"Debug: Error generating distractors with LLM: {str(e)}")
            print(f"Debug: Using fallback distractors due to error")
            fallback_verbs = ["walk", "talk", "look"]
            return fallback_verbs[:num_distractors]

    def create_multiple_choice(sentence: str, level: str) -> Optional[Dict]:
        doc = SPACY_MODEL(sentence)
        verb = None
        verb_index = None
        for i, token in enumerate(doc):
            if token.pos_ in ["VERB", "AUX"] and token.dep_ in ["ROOT", "cop"]:
                if token.text.lower() not in ["to"]:
                    verb = token.text
                    verb_index = i
                    break
        if not verb:
            print("Debug: No valid verb for multiple choice, skipping.")
            error_counts["invalid_multiple_choice"] += 1
            return None
        words = [token.text for token in doc]
        words[verb_index] = "__"
        blanked_sentence = " ".join(words).strip()
        distractors = generate_contextual_distractors(verb, sentence)
        options = distractors + [verb]
        cleaned_options = []
        for opt in options:
            clean_opt = re.sub(r'[^a-zA-Z]', '', opt.lower()).strip()
            if clean_opt and len(clean_opt.split()) == 1:
                cleaned_options.append(clean_opt)
        if len(cleaned_options) < 4:
            print("Debug: Not enough valid options, returning None")
            error_counts["invalid_multiple_choice"] += 1
            return None
        random.shuffle(cleaned_options)
        return {
            "question_text": blanked_sentence,
            "options": cleaned_options,
            "correct_answer": verb.lower()
        }

    def create_arrange_question(sentence: str, level: str) -> Optional[Dict]:
        doc = SPACY_MODEL(sentence)
        words = [token.text for token in doc if token.text != '.']
        if len(words) < 3:
            print("Debug: Sentence too short for arrange question, skipping.")
            error_counts["invalid_arrange"] += 1
            return None
        random.shuffle(words)
        if ' '.join(words).strip().lower() == sentence.strip().lower():
            random.shuffle(words)
        return {
            "question_text": "Arrange the words to form the correct sentence.",
            "words": words,
            "correct_sentence": sentence
        }

    def create_labeling_question(sentence: str, level: str) -> Optional[Dict]:
        cleaned_sentence = clean_sentence(sentence)
        if cleaned_sentence is None:
            print("Debug: Invalid sentence for labeling question, skipping.")
            return None
        doc = SPACY_MODEL(cleaned_sentence)
        root_count = sum(1 for token in doc if token.dep_ == "ROOT")
        if root_count > 1:
            print("Debug: Multiple clauses detected, skipping labeling question.")
            error_counts["multiple_clauses"] += 1
            return None
        is_passive = any(token.dep_ == "auxpass" for token in doc)
        if is_passive:
            print("Debug: Passive voice detected, skipping labeling question.")
            error_counts["passive_voice"] += 1
            return None
        constituents = {
            "subject": None,
            "verb": None,
            "object": None,
            "adjective": None,
            "adverb": None,
            "prepositional phrase": None
        }
        prep_obj = None
        main_verb = None
        for token in doc:
            if token.dep_ == "nsubj" and not constituents["subject"]:
                constituents["subject"] = token.text
                for child in token.children:
                    if child.dep_ in ["det", "compound"]:
                        constituents["subject"] = f"{child.text} {constituents['subject']}"
            if token.dep_ == "ROOT" and token.pos_ in ["VERB", "AUX"]:
                if token.pos_ == "AUX" and not main_verb:
                    for child in token.children:
                        if child.pos_ == "VERB" and child.dep_ in ["xcomp", "ccomp", "advcl"]:
                            main_verb = child.text
                            break
                if not main_verb:
                    main_verb = token.text
            if token.dep_ == "dobj" and not constituents["object"]:
                obj = token.text
                for child in token.children:
                    if child.dep_ in ["det", "amod", "compound"]:
                        obj = f"{child.text} {obj}"
                constituents["object"] = obj
            if token.dep_ in ["amod", "acomp"] and token.pos_ == "ADJ" and not constituents["adjective"]:
                constituents["adjective"] = token.text
            if token.dep_ == "advmod" and token.pos_ == "ADV" and not constituents["prepositional phrase"]:
                for child in token.head.children:
                    if child.dep_ == "prep" and child.text in ["around", "about"]:
                        constituents["prepositional phrase"] = f"{child.text}"
                        for gc in child.children:
                            if gc.dep_ == "pobj":
                                constituents["prepositional phrase"] += f" {gc.text}"
                                break
                        print(f"Debug: Relabeled '{token.text}' as prepositional phrase")
                        break
                else:
                    constituents["adverb"] = token.text
            if token.dep_ == "prep" and not constituents["prepositional phrase"]:
                for child in token.children:
                    if child.dep_ == "pobj":
                        prep_obj = child.text
                        modifiers = []
                        for grandchild in child.children:
                            if grandchild.dep_ in ["det", "amod", "nummod", "compound"]:
                                modifiers.append(grandchild.text)
                        modifiers = sorted(modifiers, key=lambda x: doc.text.find(x))
                        prep_obj = " ".join(modifiers + [prep_obj]).strip()
                        constituents["prepositional phrase"] = f"{token.text} {prep_obj}"
                        break
        constituents["verb"] = main_verb
        if constituents["verb"]:
            for contraction, expansion in CONTRACTION_MAP.items():
                if contraction in constituents["verb"].lower():
                    constituents["verb"] = expansion
        valid_constituents = [k for k, v in constituents.items() if v is not None]
        if len(valid_constituents) < 3:
            print(f"Debug: Insufficient constituents ({len(valid_constituents)}/3): " +
                  ", ".join(f"{k} ({v if v else 'None'})" for k, v in constituents.items()) +
                  ", skipping labeling question.")
            error_counts["insufficient_constituents"] += 1
            return None
        valid_values = [v.lower() for v in constituents.values() if v is not None]
        if len(set(valid_values)) != len(valid_values):
            print("Debug: Constituents are not distinct, skipping labeling question.")
            error_counts["no_valid_constituents"] += 1
            return None
        correct_labels = {k: v for k, v in constituents.items() if v is not None}
        return {
            "question_text": cleaned_sentence,
            "instruction": f"Assign {', '.join(correct_labels.keys())} to the correct words or phrases.",
            "correct_labels": correct_labels
        }

    validated_sentences = []
    target_score = cefr_to_score.get(level, 1)
    attempt_count = 0
    max_attempts = 200

    while len(validated_sentences) < num_sentences and attempt_count < max_attempts:
        attempt_count += 1
        print(f"Debug: Generating sentence attempt {attempt_count}/{max_attempts} for level {level}")
        try:
            input_text = f"### Input:\nGenerate a sentence at CEFR level {level}"
            if topic:
                input_text += f" about {topic}"
            input_text += " Please don't use to be Verb and just output the sentence\n### Response:\n"
            input_data = GENERATOR_TOKENIZER(
                input_text,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).to("cuda" if torch.cuda.is_available() else "cpu")
            max_tokens_map = {
                'A1': 30,
                'A2': 30,
                'B1': 35,
                'B2': 35,
                'C1': 50,
                'C2': 60
            }
            max_new_tokens = max_tokens_map.get(level, 30)
            print(f"Debug: Using max_new_tokens={max_new_tokens} for level {level}")
            outputs = GENERATOR_MODEL.generate(
                input_ids=input_data["input_ids"],
                attention_mask=input_data["attention_mask"],
                max_new_tokens=max_new_tokens,
                pad_token_id=GENERATOR_TOKENIZER.pad_token_id,
                do_sample=True,
                temperature=0.8,
                top_k=60,
                top_p=0.9,
            )
            if not isinstance(outputs, torch.Tensor) or outputs.numel() == 0:
                print("Debug: Invalid model output, skipping.")
                continue
            generated_text = GENERATOR_TOKENIZER.decode(outputs[0], skip_special_tokens=True)
            if not isinstance(generated_text, str) or not generated_text.strip():
                print("Debug: Invalid decoded text, skipping.")
                continue
            response_start = generated_text.find("### Response:\n") + len("### Response:\n")
            response_text = generated_text[response_start:].strip()
            response_text = re.sub(r'^Example sentence at level [A-C][1-2]: ', '', response_text, flags=re.IGNORECASE)
            original_response = response_text
            print(f"Debug: Raw model output: '{original_response}'")
            response_text = clean_sentence(response_text)
            if response_text is None:
                error_counts["no_full_stop"] += 1
                print("Debug: Sentence rejected due to missing full-stop, retrying.")
                continue
            print(f"Debug: Original sentence: '{original_response}'")
            print(f"Debug: Cleaned sentence: '{response_text}'")
            if original_response != response_text:
                print(f"Debug: Processed sentence: '{original_response}' -> '{response_text}'")
                error_counts["output_pattern_removed"] += 1
            for contraction, expansion in CONTRACTION_MAP.items():
                response_text = re.sub(rf'\b(\w+){contraction}\b', r'\1 ' + expansion, response_text, flags=re.IGNORECASE)
            doc = SPACY_MODEL(response_text)
            sents = list(doc.sents)
            first_sent = sents[0].text.strip() if sents else response_text
            if len(first_sent.split()) < 3:
                print("Debug: Sentence too short, skipping.")
                continue
            doc = SPACY_MODEL(first_sent)
            verbs = []
            for token in doc:
                if token.pos_ in ["VERB", "AUX"] and token.dep_ in ["ROOT", "cop", "aux"]:
                    if token.text.lower() == "to" and token.dep_ == "aux":
                        continue
                    if token.pos_ == "AUX" and token.dep_ == "ROOT":
                        for child in token.children:
                            if child.pos_ == "VERB" and child.dep_ in ["xcomp", "ccomp", "advcl"]:
                                verbs.append(child.text)
                                print(f"Debug: Selected main verb '{child.text}' over modal '{token.text}'")
                                break
                        else:
                            verbs.append(token.text)
                    else:
                        verbs.append(token.text)
            if not verbs and "'" in first_sent:
                contractions = [token for token in doc if "'" in token.text and token.pos_ in ["VERB", "AUX"]]
                if contractions:
                    verbs.append(contractions[0].text)
            verb = verbs[0] if verbs else None
            if not verb:
                error_counts["no_valid_verb"] += 1
                print("Debug: No valid verb found, skipping.")
                continue
            predicted_score = predict_sentiment(first_sent)
            if abs(predicted_score - target_score) > 1:
                error_counts["cefr_mismatch"] += 1
                print("Debug: Sentence CEFR level mismatch, skipping.")
                continue
            if any(existing["sentence"] == first_sent for existing in validated_sentences):
                error_counts["duplicate_sentence"] += 1
                print("Debug: Duplicate sentence, skipping.")
                continue
            sentence_data = {
                "sentence": first_sent,
                "verb": verb,
                "level": level,
                "for_user": user_id
            }
            if "selection" in question_types:
                selection_question = create_selection_question(first_sent, level)
                sentence_data["selection_question"] = selection_question
            if "labeling" in question_types:
                labeling_question = create_labeling_question(first_sent, level)
                sentence_data["labeling_question"] = labeling_question
            if "fill_in_blank" in question_types:
                fill_in_blank_question = create_fill_in_blank(first_sent, level)
                sentence_data["fill_in_blank_question"] = fill_in_blank_question
            if "multiple_choice" in question_types:
                multiple_choice_question = create_multiple_choice(first_sent, level)
                sentence_data["multiple_choice_question"] = multiple_choice_question
            if "arrange" in question_types:
                arrange_question = create_arrange_question(first_sent, level)
                sentence_data["arrange_question"] = arrange_question
            if not any([sentence_data.get(qt + "_question") for qt in question_types]):
                print("Debug: No valid questions generated, skipping.")
                continue
            validated_sentences.append(sentence_data)
            print(f"Debug: Added valid sentence: '{first_sent}'")
        except Exception as e:
            print(f"Error generating sentence: {str(e)}")
            continue

    if not validated_sentences:
        print(f"Warning: No valid sentences generated after {attempt_count} attempts.")

    print("\nError Statistics:")
    print(f"Duplicate sentence = {error_counts['duplicate_sentence']} Case")
    print(f"Sentence CEFR level mismatch = {error_counts['cefr_mismatch']} Case")
    print(f"No valid verb found = {error_counts['no_valid_verb']} Case")
    print(f"No valid constituents = {error_counts['no_valid_constituents']} Case")
    print(f"Insufficient constituents = {error_counts['insufficient_constituents']} Case")
    print(f"Multiple clauses = {error_counts['multiple_clauses']} Case")
    print(f"Passive voice = {error_counts['passive_voice']} Case")
    print(f"Duplicate selection sentences = {error_counts['duplicate_selection']} Case")
    print(f"Invalid incorrect sentences = {error_counts['invalid_incorrect_sentence']} Case")
    print(f"Invalid fill-in-the-blank = {error_counts['invalid_fill_in_blank']} Case")
    print(f"Invalid multiple choice = {error_counts['invalid_multiple_choice']} Case")
    print(f"Invalid arrange questions = {error_counts['invalid_arrange']} Case")
    print(f"No full-stop in sentence = {error_counts['no_full_stop']} Case")

    return validated_sentences

# Evaluation functions
def evaluate_selection(selected: str, correct: str) -> Dict:
    selected = clean_sentence(selected)
    if selected is None:
        return {
            "is_correct": False,
            "score": 0.0,
            "message": "Invalid selected sentence: no full-stop found."
        }
    correct = clean_sentence(correct)
    if correct is None:
        return {
            "is_correct": False,
            "score": 0.0,
            "message": "Invalid correct sentence: no full-stop found."
        }
    is_correct = selected.strip() == correct.strip()
    score = 1.0 if is_correct else 0.0
    return {
        "is_correct": is_correct,
        "score": score,
        "message": "Selection is correct." if is_correct else "Selection is incorrect."
    }

def evaluate_labeling(sentence: str, user_labels: Dict[str, str]) -> Dict:
    global SPACY_MODEL
    cleaned_sentence = clean_sentence(sentence)
    if cleaned_sentence is None:
        return {
            "is_correct": False,
            "score": 0.0,
            "message": "Invalid sentence: no full-stop found."
        }
    doc = SPACY_MODEL(cleaned_sentence)
    correct_labels = {
        "subject": None,
        "verb": None,
        "object": None,
        "adjective": None,
        "adverb": None,
        "prepositional phrase": None
    }
    prep_obj = None
    main_verb = None
    for token in doc:
        if token.dep_ == "nsubj" and not correct_labels["subject"]:
            correct_labels["subject"] = token.text
            for child in token.children:
                if child.dep_ in ["det", "compound"]:
                    correct_labels["subject"] = f"{child.text} {correct_labels['subject']}"
        if token.dep_ == "ROOT" and token.pos_ in ["VERB", "AUX"]:
            if token.pos_ == "AUX" and not main_verb:
                for child in token.children:
                    if child.pos_ == "VERB" and child.dep_ in ["xcomp", "ccomp", "advcl"]:
                        main_verb = child.text
                        break
            if not main_verb:
                main_verb = token.text
        if token.dep_ == "dobj" and not correct_labels["object"]:
            obj = token.text
            for child in token.children:
                if child.dep_ in ["det", "amod", "compound"]:
                    obj = f"{child.text} {obj}"
            correct_labels["object"] = obj
        if token.dep_ in ["amod", "acomp"] and token.pos_ == "ADJ" and not correct_labels["adjective"]:
            correct_labels["adjective"] = token.text
        if token.dep_ == "advmod" and token.pos_ == "ADV" and not correct_labels["prepositional phrase"]:
            for child in token.head.children:
                if child.dep_ == "prep" and child.text in ["around", "about"]:
                    correct_labels["prepositional phrase"] = f"{child.text}"
                    for gc in child.children:
                        if gc.dep_ == "pobj":
                            correct_labels["prepositional phrase"] += f" {gc.text}"
                            break
                    print(f"Debug: Relabeled '{token.text}' as prepositional phrase")
                    break
            else:
                correct_labels["adverb"] = token.text
        if token.dep_ == "prep" and not correct_labels["prepositional phrase"]:
            for child in token.children:
                if child.dep_ == "pobj":
                    prep_obj = child.text
                    for grandchild in child.children:
                        if grandchild.dep_ in ["det", "amod"]:
                            prep_obj = f"{grandchild.text} {prep_obj}"
                    correct_labels["prepositional phrase"] = f"{token.text} {prep_obj}"
                    break
    correct_labels["verb"] = main_verb
    if correct_labels["verb"]:
        for contraction, expansion in {
            "'ve": "have",
            "'s": "is",
            "'re": "are",
            "'m": "am",
            "'ll": "will",
            "'d": "had"
        }.items():
            if contraction in correct_labels["verb"].lower():
                correct_labels["verb"] = expansion
    valid_correct_labels = {k: v for k, v in correct_labels.items() if v is not None}
    if not valid_correct_labels:
        return {
            "is_correct": False,
            "score": 0.0,
            "message": "No valid constituents found in sentence."
        }
    score_per_label = 1.0 / len(valid_correct_labels)
    is_correct = all(
        user_labels.get(key, "").strip().lower() == value.strip().lower()
        for key, value in valid_correct_labels.items()
    )
    score = sum(
        score_per_label
        for key, value in valid_correct_labels.items()
        if user_labels.get(key, "").strip().lower() == value.strip().lower()
    )
    return {
        "is_correct": is_correct,
        "score": round(score, 2),
        "message": "Labeling is correct." if is_correct else "Labeling is incorrect."
    }

def evaluate_fill_in_blank(user_answer: str, correct_answer: str) -> Dict:
    is_correct = user_answer.strip().lower() == correct_answer.strip().lower()
    score = 1.0 if is_correct else 0.0
    return {
        "is_correct": is_correct,
        "score": score,
        "message": "Answer is correct." if is_correct else "Answer is incorrect."
    }

def evaluate_multiple_choice(selected_option: str, correct_answer: str) -> Dict:
    is_correct = selected_option.strip().lower() == correct_answer.strip().lower()
    score = 1.0 if is_correct else 0.0
    return {
        "is_correct": is_correct,
        "score": score,
        "message": "Selection is correct." if is_correct else "Selection is incorrect."
    }

def evaluate_arrange(user_arrangement: List[str], correct_sentence: str) -> Dict:
    user_sentence = " ".join(user_arrangement).strip()
    correct_sentence = clean_sentence(correct_sentence)
    if correct_sentence is None:
        return {
            "is_correct": False,
            "score": 0.0,
            "message": "Invalid correct sentence: no full-stop found."
        }
    is_correct = user_sentence.lower() == correct_sentence.lower()
    score = 1.0 if is_correct else 0.0
    return {
        "is_correct": is_correct,
        "score": score,
        "message": "Arrangement is correct." if is_correct else "Arrangement is incorrect."
    }

# API endpoints
@app.post("/generate_sentences", response_model=SentenceResponse)
async def generate_sentences(request: Request, sentence_request: SentenceRequest):
    start_time = time.time()
    try:
        level = sentence_request.level
        num_sentences = sentence_request.num_sentences
        user_id = sentence_request.user_id
        topic = sentence_request.topic
        question_types = sentence_request.question_types
        valid_question_types = ["selection", "labeling", "fill_in_blank", "multiple_choice", "arrange"]
        if not all(qt in valid_question_types for qt in question_types):
            raise HTTPException(status_code=400, detail=f"Invalid question types. Must be subset of: {valid_question_types}")
        if level not in ["A1", "A2", "B1", "B2", "C1", "C2"]:
            raise HTTPException(status_code=400, detail="Invalid level. Must be one of: A1, A2, B1, B2, C1, C2")
        if num_sentences < 1 or num_sentences > 10:
            raise HTTPException(status_code=400, detail="Number of sentences must be between 1 and 10")
        sentences = generate_sentences_internal(level, num_sentences, user_id, topic, question_types)
        processing_time = time.time() - start_time
        current_time = datetime.datetime.now().isoformat()
        if not sentences:
            print("Debug: No sentences generated, returning empty response")
            return {
                "success": False,
                "sentences": [],
                "message": "Failed to generate sentences. Please try again.",
                "processing_time": processing_time,
                "timestamp": current_time
            }
        response = {
            "success": True,
            "sentences": sentences,
            "message": f"Generated {len(sentences)} sentences for level {level}" +
                      (f" on topic '{topic}'" if topic else "") +
                      f" with question types {question_types}",
            "processing_time": processing_time,
            "timestamp": current_time
        }
        print(f"Debug: Sending response: {response}")
        return response
    except HTTPException:
        raise
    except Exception as e:
        processing_time = time.time() - start_time
        current_time = datetime.datetime.now().isoformat()
        print(f"API error: {str(e)}")
        return {
            "success": False,
            "sentences": [],
            "message": f"Internal server error: {str(e)}",
            "processing_time": processing_time,
            "timestamp": current_time
        }

@app.post("/evaluate_selection", response_model=SelectionResponse)
async def evaluate_selection(request: Request, selection_request: SelectionRequest):
    start_time = time.time()
    try:
        result = evaluate_selection(
            selection_request.selected_sentence,
            selection_request.correct_sentence
        )
        processing_time = time.time() - start_time
        current_time = datetime.datetime.now().isoformat()
        return {
            "success": True,
            "is_correct": result["is_correct"],
            "score": result["score"],
            "message": result["message"],
            "processing_time": processing_time,
            "timestamp": current_time
        }
    except Exception as e:
        processing_time = time.time() - start_time
        current_time = datetime.datetime.now().isoformat()
        return {
            "success": False,
            "is_correct": False,
            "score": 0.0,
            "message": f"Error evaluating selection: {str(e)}",
            "processing_time": processing_time,
            "timestamp": current_time
        }

def run_server(port: int = 8000, max_attempts: int = 5):
    import socket
    for attempt in range(max_attempts):
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                s.bind(("0.0.0.0", port))
            config = uvicorn.Config(
                app,
                host="0.0.0.0",
                port=port,
                log_level="info",
                timeout_keep_alive=30
            )
            server = uvicorn.Server(config)
            print(f"Starting server on port {port}...")
            asyncio.run(server.serve())
            break
        except OSError as e:
            if "address already in use" in str(e).lower() and attempt < max_attempts - 1:
                print(f"Port {port} in use, trying port {port + 1}...")
                port += 1
                time.sleep(1)
            else:
                print(f"Failed to start server: {str(e)}")
                raise
        except Exception as e:
            print(f"Server error: {str(e)}")
            raise

def start_ngrok(port: int) -> Optional[str]:
    try:
        ngrok.kill()
        ngrok.set_auth_token("2oFl98p58vTuHbdQu5m2t1exmDW_5MXsrmeuZtv9J5hTscGzb")
        public_url = ngrok.connect(port, bind_tls=True).public_url
        print(f"Ngrok tunnel created: {public_url}")
        return public_url
    except Exception as e:
        print(f"Ngrok error: {str(e)}")
        print("Continuing without ngrok - you'll need to access the API locally")
        return None

def start_application():
    try:
        load_models()
    except Exception as e:
        print(f"Failed to load models: {str(e)}")
        return
    port = 8000
    max_attempts = 5
    attempt = 0
    while attempt < max_attempts:
        try:
            server_thread = threading.Thread(target=run_server, args=(port,))
            server_thread.daemon = True
            server_thread.start()
            time.sleep(2)
            public_url = start_ngrok(port)
            if public_url:
                print(f"\nAPI is available at: {public_url}/generate_sentences")
            else:
                print(f"\nAPI is running locally at: http://localhost:{port}/generate_sentences")
            while True:
                time.sleep(1)
        except KeyboardInterrupt:
            print("\nShutting down server...")
            ngrok.kill()
            import os
            os._exit(0)
        except Exception as e:
            print(f"Application error: {str(e)}")
            ngrok.kill()
            attempt += 1
            port += 1
            time.sleep(2)
            if attempt < max_attempts:
                print(f"Retrying with port {port}...")

if __name__ == "__main__":
    start_application()